# Titanic — Machine Learning from Disaster

**Goal:** Predict which passengers survived the Titanic shipwreck (binary classification).

**Metric:** Accuracy (percentage of correct predictions).

**Kaggle link:** https://www.kaggle.com/competitions/titanic

## 1. Setup & Imports

In [ ]:
# Standard library
import warnings
from pathlib import Path

# Visualization
import matplotlib.pyplot as plt

# Data manipulation
import numpy as np
import pandas as pd
import seaborn as sns

# Machine Learning

# Settings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

%matplotlib inline

## 2. Configuration

In [ ]:
# Project paths
DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")
MODELS_DIR = Path("../outputs/models")
SUBMISSIONS_DIR = Path("../outputs/submissions")

# Competition settings
COMPETITION_NAME = "titanic"
TARGET_COL = "Survived"
RANDOM_STATE = 42
TEST_SIZE = 0.2

## 3. Load Data

In [ ]:
# TODO: Update filenames to match your competition data
train_df = pd.read_csv(DATA_RAW / "train.csv")
test_df = pd.read_csv(DATA_RAW / "test.csv")

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
print(f"\nTrain columns: {list(train_df.columns)}")

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Overview
train_df.head()

In [ ]:
# Data types and non-null counts
train_df.info()

In [ ]:
# Statistical summary
train_df.describe()

In [ ]:
# Missing values
missing = train_df.isnull().sum()
missing_pct = (missing / len(train_df)) * 100
missing_df = pd.DataFrame({"count": missing, "percentage": missing_pct})
print(missing_df[missing_df["count"] > 0].sort_values("percentage", ascending=False))

In [ ]:
# Target distribution
survived_counts = train_df[TARGET_COL].value_counts()
survived_pct = train_df[TARGET_COL].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
survived_counts.plot(kind="bar", ax=axes[0], color=["#e74c3c", "#2ecc71"])
axes[0].set_title(f"Target Distribution: {TARGET_COL}")
axes[0].set_ylabel("Count")
axes[0].set_xticklabels(["Did not survive (0)", "Survived (1)"], rotation=0)

# Percentage pie chart
axes[1].pie(
    survived_pct,
    labels=["Did not survive (0)", "Survived (1)"],
    autopct="%1.1f%%",
    colors=["#e74c3c", "#2ecc71"],
    startangle=90,
)
axes[1].set_title("Survival Rate")

plt.tight_layout()
plt.show()

print(f"Survival rate: {survived_pct.iloc[1]:.1f}% survived, {survived_pct.iloc[0]:.1f}% did not survive")

In [ ]:
# Correlation heatmap (numeric features only)
numeric_cols = train_df.select_dtypes(include=[np.number]).columns
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(train_df[numeric_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
ax.set_title("Correlation Matrix")
plt.tight_layout()
plt.show()

### Survival Rate by Sex

In [ ]:
# Survival rate by Sex
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.countplot(data=train_df, x="Sex", hue=TARGET_COL, ax=axes[0], palette=["#e74c3c", "#2ecc71"])
axes[0].set_title("Survival Count by Sex")
axes[0].legend(title="Survived", labels=["No", "Yes"])

sex_survival = train_df.groupby("Sex")[TARGET_COL].mean() * 100
sex_survival.plot(kind="bar", ax=axes[1], color=["#3498db", "#e67e22"])
axes[1].set_title("Survival Rate by Sex (%)")
axes[1].set_ylabel("Survival Rate (%)")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

print(sex_survival.to_string())

### Survival Rate by Passenger Class

In [ ]:
# Survival rate by Pclass
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.countplot(data=train_df, x="Pclass", hue=TARGET_COL, ax=axes[0], palette=["#e74c3c", "#2ecc71"])
axes[0].set_title("Survival Count by Passenger Class")
axes[0].legend(title="Survived", labels=["No", "Yes"])

pclass_survival = train_df.groupby("Pclass")[TARGET_COL].mean() * 100
pclass_survival.plot(kind="bar", ax=axes[1], color=["#f1c40f", "#3498db", "#e74c3c"])
axes[1].set_title("Survival Rate by Passenger Class (%)")
axes[1].set_ylabel("Survival Rate (%)")
axes[1].set_xticklabels(["1st", "2nd", "3rd"], rotation=0)

plt.tight_layout()
plt.show()

print(pclass_survival.to_string())

### Age Distribution by Survival

In [ ]:
# Age distribution by survival status
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overlapping histograms
for survived, color, label in [(0, "#e74c3c", "Did not survive"), (1, "#2ecc71", "Survived")]:
    subset = train_df[train_df[TARGET_COL] == survived]["Age"].dropna()
    axes[0].hist(subset, bins=30, alpha=0.6, color=color, label=label)
axes[0].set_title("Age Distribution by Survival")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Count")
axes[0].legend()

# Box plot
sns.boxplot(data=train_df, x=TARGET_COL, y="Age", ax=axes[1], palette=["#e74c3c", "#2ecc71"])
axes[1].set_title("Age Distribution by Survival (Box Plot)")
axes[1].set_xticklabels(["Did not survive", "Survived"])

plt.tight_layout()
plt.show()

print(f"Age missing: {train_df['Age'].isnull().sum()} ({train_df['Age'].isnull().mean() * 100:.1f}%)")
print("\nAge stats by survival:")
print(train_df.groupby(TARGET_COL)["Age"].describe())

### Embarked, Fare, and Family Size Analysis

In [ ]:
# Embarked, Fare, and Family analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Survival by Embarked
sns.countplot(data=train_df, x="Embarked", hue=TARGET_COL, ax=axes[0, 0], palette=["#e74c3c", "#2ecc71"])
axes[0, 0].set_title("Survival by Embarkation Port")
axes[0, 0].legend(title="Survived", labels=["No", "Yes"])

embarked_survival = train_df.groupby("Embarked")[TARGET_COL].mean() * 100
print("Survival rate by Embarked port:")
print(embarked_survival.to_string(), "\n")

# Fare distribution
sns.boxplot(data=train_df, x=TARGET_COL, y="Fare", ax=axes[0, 1], palette=["#e74c3c", "#2ecc71"])
axes[0, 1].set_title("Fare Distribution by Survival")
axes[0, 1].set_xticklabels(["Did not survive", "Survived"])

# Family size (SibSp + Parch)
train_df["FamilySize"] = train_df["SibSp"] + train_df["Parch"] + 1
family_survival = train_df.groupby("FamilySize")[TARGET_COL].mean() * 100

family_survival.plot(kind="bar", ax=axes[1, 0], color="#3498db")
axes[1, 0].set_title("Survival Rate by Family Size (%)")
axes[1, 0].set_xlabel("Family Size")
axes[1, 0].set_ylabel("Survival Rate (%)")

# Alone vs with family
train_df["IsAlone"] = (train_df["FamilySize"] == 1).astype(int)
alone_survival = train_df.groupby("IsAlone")[TARGET_COL].mean() * 100
alone_survival.index = ["With family", "Alone"]
alone_survival.plot(kind="bar", ax=axes[1, 1], color=["#2ecc71", "#e74c3c"])
axes[1, 1].set_title("Survival Rate: Alone vs With Family (%)")
axes[1, 1].set_ylabel("Survival Rate (%)")
axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

print(f"\nFamily size distribution:\n{train_df['FamilySize'].value_counts().sort_index().to_string()}")

### Cabin Feature Analysis

In [ ]:
# Cabin feature analysis (77% missing)
cabin_missing_pct = train_df["Cabin"].isnull().mean() * 100
has_cabin = train_df["Cabin"].notna().astype(int)

fig, ax = plt.subplots(figsize=(8, 5))
cabin_survival = train_df.groupby(has_cabin)[TARGET_COL].mean() * 100
cabin_survival.index = ["No cabin info", "Has cabin info"]
cabin_survival.plot(kind="bar", ax=ax, color=["#e74c3c", "#2ecc71"])
ax.set_title("Survival Rate by Cabin Info Availability (%)")
ax.set_ylabel("Survival Rate (%)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

print(f"Cabin missing: {train_df['Cabin'].isnull().sum()} / {len(train_df)} ({cabin_missing_pct:.1f}%)")
print(f"Survival rate with cabin info: {cabin_survival['Has cabin info']:.1f}%")
print(f"Survival rate without cabin info: {cabin_survival['No cabin info']:.1f}%")
print("\nNote: Cabin availability correlates with class/fare — likely a proxy for wealth.")

### EDA Summary

In [ ]:
# EDA key findings summary
print("=" * 60)
print("EDA KEY FINDINGS")
print("=" * 60)
print("""
1. IMBALANCED TARGET: ~38% survived, ~62% did not survive.

2. SEX is the strongest predictor:
   - Female survival rate ~74%, male ~19%.

3. PCLASS matters significantly:
   - 1st class ~63%, 2nd ~47%, 3rd ~24%.

4. AGE: Children had higher survival rates.
   - ~20% of Age values are missing — needs imputation.

5. FARE: Survivors paid higher fares on average (correlated with Pclass).

6. FAMILY SIZE: Solo travelers and very large families had lower survival.
   - Sweet spot around 2-4 family members.

7. EMBARKED: Cherbourg (C) passengers had highest survival rate.

8. CABIN: 77% missing. Having cabin info correlates with higher survival
   (proxy for wealth/class). Consider binary has_cabin feature.
""")

## 5. Data Cleaning

In [ ]:
# TODO: Handle missing values
# Examples:
# train_df["column"].fillna(train_df["column"].median(), inplace=True)
# train_df.dropna(subset=["important_column"], inplace=True)

# TODO: Fix data types
# train_df["column"] = train_df["column"].astype("category")

# TODO: Remove duplicates if needed
# train_df.drop_duplicates(inplace=True)

## 6. Feature Engineering

In [ ]:
# TODO: Create new features
# Examples:
# train_df["new_feature"] = train_df["col_a"] / train_df["col_b"]
# train_df["is_weekend"] = train_df["day_of_week"].isin([5, 6]).astype(int)

# TODO: Encode categorical variables
# le = LabelEncoder()
# train_df["category_encoded"] = le.fit_transform(train_df["category"])

# TODO: Apply same transformations to test_df!

## 7. Modeling

In [ ]:
# Prepare features and target
# TODO: Select your feature columns
# feature_cols = ["col1", "col2", "col3"]
# X = train_df[feature_cols]
# y = train_df[TARGET_COL]

# Train/validation split
# X_train, X_val, y_train, y_val = train_test_split(
#     X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
# )
# print(f"Train: {X_train.shape}, Validation: {X_val.shape}")

In [ ]:
# Baseline model
# TODO: Choose your baseline model

# For classification:
# model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
# model.fit(X_train, y_train)

# Cross-validation
# cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")
# print(f"CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## 8. Evaluation

In [ ]:
# Predictions on validation set
# y_pred = model.predict(X_val)

# For classification:
# print(f"Accuracy: {accuracy_score(y_val, y_pred):.4f}")
# print(f"\nClassification Report:")
# print(classification_report(y_val, y_pred))

# Confusion matrix
# fig, ax = plt.subplots(figsize=(8, 6))
# cm = confusion_matrix(y_val, y_pred)
# sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
# ax.set_title("Confusion Matrix")
# ax.set_ylabel("Actual")
# ax.set_xlabel("Predicted")
# plt.tight_layout()
# plt.show()

In [ ]:
# Feature importance
# TODO: Uncomment after training a tree-based model

# importance_df = pd.DataFrame({
#     "feature": feature_cols,
#     "importance": model.feature_importances_
# }).sort_values("importance", ascending=False)

# fig, ax = plt.subplots(figsize=(10, 6))
# sns.barplot(data=importance_df, x="importance", y="feature", ax=ax)
# ax.set_title("Feature Importance")
# plt.tight_layout()
# plt.show()

## 9. Submission

In [ ]:
# Generate predictions on test set
# TODO: Apply same preprocessing to test_df
# X_test = test_df[feature_cols]
# test_predictions = model.predict(X_test)

# Create submission file
# TODO: Adapt columns to match competition requirements
# submission = pd.DataFrame({
#     "Id": test_df["Id"],
#     TARGET_COL: test_predictions
# })

# submission.to_csv(SUBMISSIONS_DIR / "submission.csv", index=False)
# print(f"Submission saved to {SUBMISSIONS_DIR / 'submission.csv'}")
# print(f"Shape: {submission.shape}")
# submission.head()